# Scraping Google search results: AI Copyright & Governance

---
## 0. Install Dependencies

In [ ]:
%pip install google-search-results pandas python-dotenv requests trafilatura youtube_transcript_api


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


---
## 1. Imports & Configuration

In [ ]:
from serpapi import GoogleSearch
from youtube_transcript_api import YouTubeTranscriptApi
from concurrent.futures import ThreadPoolExecutor, as_completed
import trafilatura
import pandas as pd
from dotenv import load_dotenv
import os
import re
import time

load_dotenv()
# Accept the common secret names; the value is never printed.
SERP_API_KEY = next(
    (os.environ[k].strip() for k in ("SERPAPI_KEY", "SERPAPI_API_KEY", "SERPAPI")
     if os.environ.get(k, "").strip()),
    None,
)
if not SERP_API_KEY:
    raise ValueError("Set SERPAPI_KEY (or SERPAPI_API_KEY / SERPAPI) in the environment or .env")

# Max number of articles to scrape
MAX_NUM_RESULTS = 10

# Pages with less text are usually login walls or bot checks, not articles
MIN_TEXT_CHARS = 500

# Search results can include generic pages (e.g. Wikipedia "Artificial intelligence",
# vendor homepages). Keep only text that discusses the research topic.
# Adjust TOPIC_TERMS when you research a different topic.
TOPIC_TERMS = re.compile(
    r"copyright|intellectual property|\bpatent|trademark|fair use|infring|licens",
    re.IGNORECASE,
)
MIN_TOPIC_MENTIONS_PER_1K_WORDS = 5

---
## 2. SerpApi - Collecting article & video URLs

In [ ]:
def collect_search_results(queries: list[str],
                           num_results: int = MAX_NUM_RESULTS) -> tuple[pd.DataFrame, list[dict]]:
    """
    Search Google for each query and collect:
    - organic_results  → articles (title, snippet, url, source, date)
    - inline_videos    → YouTube videos (title, channel, url)

    Returns:
        articles_df  : DataFrame of all article metadata
        raw_results  : list of raw SerpAPI result dicts (just for inspection)
    """
    article_rows = []
    video_rows   = []
    raw_results  = []

    for query in queries:
        print(f"\n🔍 Searching: '{query}'")

        params = {
            "engine":        "google",
            "q":             query,
            "google_domain": "google.com",
            "hl":            "en",
            "gl":            "us",
            "api_key":       SERP_API_KEY
        }

        search  = GoogleSearch(params)
        results = search.get_dict()
        raw_results.append({"query": query, "results": results})

        # ── Organic articles ──
        organic = results.get("organic_results", [])
        if not organic:
            print(f"  ⚠️  No organic_results for: {query}")
        else:
            for article in organic[:num_results]:
                article_rows.append({
                    "query":   query,
                    "title":   article.get("title"),
                    "snippet": article.get("snippet"),
                    "source":  article.get("source"),
                    "date":    article.get("date"),
                    "url":     article.get("link"),
                    "type":    "article"
                })
            print(f"  ✅ {len(organic[:num_results])} articles collected")

        # ── Inline videos ──
        videos = results.get("inline_videos", [])
        if videos:
            for video in videos:
                video_rows.append({
                    "query":   query,
                    "title":   video.get("title"),
                    "snippet": None,
                    "source":  video.get("channel"),
                    "date":    None,
                    "url":     video.get("link"),
                    "type":    "video"
                })
            print(f"  ✅ {len(videos)} videos collected")

    # Combine into single DataFrame
    all_rows = article_rows + video_rows
    df = pd.DataFrame(all_rows)

    if df.empty:
        print("\n⚠️  No results collected across all queries.")
        return df, raw_results

    # Remove duplicates by URL
    df = df.drop_duplicates(subset="url").reset_index(drop=True)

    return df, raw_results

In [ ]:
# ── Collect urls from search query ──
df, raw = collect_search_results(["AI intellectual property", 
                                  "copyright Generative AI"])
df.head()


🔍 Searching: 'AI intellectual property'
  ✅ 8 articles collected

🔍 Searching: 'copyright Generative AI'
  ✅ 8 articles collected


,query,title,snippet,source,date,url,type
0,AI intellectual property,‎Google Gemini,"Meet Gemini, Google's AI assistant. Get help w...",‎Google Gemini,NaN,https://gemini.google.com/,article
1,AI intellectual property,Artificial intelligence,Artificial intelligence (AI) is the capability...,Wikipedia,NaN,https://en.wikipedia.org/wiki/Artificial_intel...,article
2,AI intellectual property,Perplexity AI,Perplexity is a free AI-powered answer engine ...,Perplexity,NaN,https://www.perplexity.ai/,article
3,AI intellectual property,"ChatGPT: Chat, Work, Create & Code with AI","Use ChatGPT to answer questions, write, create...",ChatGPT,NaN,https://chatgpt.com/,article
4,AI intellectual property,What is Artificial Intelligence (AI)?,AI is a field of computer science focused on c...,Google Cloud,NaN,https://cloud.google.com/learn/what-is-artific...,article


In [5]:
raw

[{'query': 'AI intellectual property',
  'results': {'search_metadata': {'id': '6adf57229616211fb0cf2bba',
    'status': 'Success',
    'json_endpoint': 'https://serpapi.com/searches/lGWMf4PK3eS7uB3-3xm4YoMdlASp6si5SYyH1PCPXvc/6adf57229616211fb0cf2bba.json',
    'markdown_endpoint': 'https://serpapi.com/searches/lGWMf4PK3eS7uB3-3xm4YoMdlASp6si5SYyH1PCPXvc/6adf57229616211fb0cf2bba.md',
    'pixel_position_endpoint': 'https://serpapi.com/searches/lGWMf4PK3eS7uB3-3xm4YoMdlASp6si5SYyH1PCPXvc/6adf57229616211fb0cf2bba.json_with_pixel_position',
    'created_at': '2026-10-26 13:35:30 UTC',
    'processed_at': '2026-10-26 13:35:30 UTC',
    'google_url': 'https://www.google.com/search?q=AI+intellectual+property&oq=AI+intellectual+property&hl=en&gl=us&sourceid=chrome&ie=UTF-8',
    'raw_html_file': 'https://serpapi.com/searches/lGWMf4PK3eS7uB3-3xm4YoMdlASp6si5SYyH1PCPXvc/6adf57229616211fb0cf2bba.html',
    'total_time_taken': 0.44},
   'search_parameters': {'engine': 'google',
    'q': 'AI inte

---
## 3. Scraping texts from articles and videos

In [6]:
# Define scrapers
def _scrape_url(url: str) -> dict:
    """Scrape clean article text from a URL using Trafilatura."""
    try:
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            return {"url": url, "full_text": None,
                    "status": "failed_download"}

        text = trafilatura.extract(
            downloaded,
            include_comments=False,
            include_tables=True,
            no_fallback=False,
            favor_precision=False,
            deduplicate=True
        )
        text = text.strip() if text else None
        if not text:
            status = "failed_extraction"
        elif len(text) < MIN_TEXT_CHARS:
            status = "too_short"
        else:
            status = "success"
        return {
            "url":       url,
            "full_text": text if status == "success" else None,
            "status":    status
        }
    except Exception as e:
        return {"url": url, "full_text": None,
                "status": f"error: {str(e)}"}


def _get_youtube_id(url: str) -> str | None:
    """Extract YouTube video ID from URL."""
    patterns = [
        r"youtube\.com/watch\?v=([a-zA-Z0-9_-]{11})",
        r"youtube\.com/shorts/([a-zA-Z0-9_-]{11})",
        r"youtu\.be/([a-zA-Z0-9_-]{11})"
    ]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    return None


def _get_transcript(video_id: str) -> str | None:
    """Fetch YouTube transcript from video ID."""
    try:
        # youtube-transcript-api >= 1.0 replaced get_transcript() with fetch().
        transcript = YouTubeTranscriptApi().fetch(
            video_id,
            languages=["en", "en-US", "en-GB"]
        )
        return " ".join(snippet.text for snippet in transcript)
    except Exception:
        return None


# ═══════════════════════════════════════════════════════
# ENRICHMENT
# ═══════════════════════════════════════════════════════

def enrich_search_results(df: pd.DataFrame,
                         max_workers: int = 5,
                         delay: float = 1.0) -> pd.DataFrame:
    """
    Enrich a DataFrame from collect_search_results() with full text content.
    - Rows with type='article' → scraped via Trafilatura
    - Rows with type='video'   → transcript via YouTube Transcript API

    Args:
        df:          DataFrame returned by collect_articles()
        max_workers: concurrent threads for article scraping
        delay:       seconds between requests per thread

    Returns:
        Enriched DataFrame with added columns:
        - full_text  : scraped article text OR video transcript
        - video_id   : YouTube video ID (videos only)
        - status     : success / failed_download / no_transcript / etc.
    """
    if df.empty:
        print("⚠️  Empty DataFrame — nothing to enrich.")
        return df

    df = df.copy()
    df["full_text"] = None
    df["video_id"]  = None
    df["status"]    = "pending"

    # ── Split by type ──
    article_mask = df["type"] == "article"
    video_mask   = df["type"] == "video"

    article_df = df[article_mask].copy()
    video_df   = df[video_mask].copy()

    # ── Scrape articles ──
    if not article_df.empty:
        print(f"\n📰 Scraping {len(article_df)} articles...")

        url_to_result = {}

        def fetch_with_delay(url):
            time.sleep(delay)
            return _scrape_url(url)

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {
                executor.submit(fetch_with_delay, row["url"]): idx
                for idx, row in article_df.iterrows()
            }
            for i, future in enumerate(as_completed(futures)):
                result = future.result()
                url_to_result[result["url"]] = result
                icon = "✅" if result["status"] == "success" else "❌"
                source = df.loc[futures[future], "source"] or "Unknown"
                print(f"  {icon} [{i+1}/{len(article_df)}] "
                      f"{source} — {result['status']}")

        # Map results back
        for idx, row in article_df.iterrows():
            r = url_to_result.get(row["url"], {})
            df.at[idx, "full_text"] = r.get("full_text")
            df.at[idx, "status"]    = r.get("status", "unknown")

        success = sum(
            1 for r in url_to_result.values()
            if r["status"] == "success"
        )
        print(f"\n  📊 Articles: {success}/{len(article_df)} "
              f"scraped successfully")

    # ── Fetch video transcripts ──
    if not video_df.empty:
        print(f"\n🎬 Fetching transcripts for "
              f"{len(video_df)} videos...")

        for idx, row in video_df.iterrows():
            url      = row["url"]
            vid_id   = _get_youtube_id(url)
            transcript = None

            if vid_id:
                transcript = _get_transcript(vid_id)

            status = "success" if transcript else "no_transcript"
            icon   = "✅" if transcript else "❌"

            df.at[idx, "video_id"]  = vid_id
            df.at[idx, "full_text"] = transcript
            df.at[idx, "status"]    = status

            title = (row["title"] or "")[:50]
            print(f"  {icon} {title}... [{status}]")

        success = sum(
            1 for idx in video_df.index
            if df.at[idx, "status"] == "success"
        )
        print(f"\n  📊 Videos: {success}/{len(video_df)} "
              f"transcripts fetched")

    return df

In [7]:
# ── Enrich data with full texts from articles and videos ──
df = enrich_search_results(df)

# ── Mark scraped pages that are not about the research topic ──
def topic_mentions_per_1k_words(text) -> float:
    if not isinstance(text, str) or not text.strip():
        return 0.0
    return 1000 * len(TOPIC_TERMS.findall(text)) / len(text.split())

df["topic_mentions_per_1k"] = df["full_text"].apply(topic_mentions_per_1k_words).round(2)
off_topic = (df["status"] == "success") & (
    df["topic_mentions_per_1k"] < MIN_TOPIC_MENTIONS_PER_1K_WORDS
)
df.loc[off_topic, "status"] = "off_topic"
print(f"\n🧹 Removed {off_topic.sum()} off-topic pages:")
for _, row in df[off_topic].iterrows():
    print(f"  ❌ {row['source']} — {row['title']} "
          f"({row['topic_mentions_per_1k']} topic mentions per 1,000 words)")

# ── Filter to only successfully scraped, on-topic content ──
df_clean = df[df["status"] == "success"].reset_index(drop=True)
print(f"\n📊 Kept {len(df_clean)} on-topic documents")

# ── Inspect results ──
df_clean[["query", "source", "title", "topic_mentions_per_1k"]]



📰 Scraping 16 articles...


  ❌ [1/16] Perplexity — failed_extraction
  ❌ [2/16] ‎Google Gemini — too_short
  ❌ [3/16] ChatGPT — failed_extraction


  ✅ [4/16] Google Cloud — success


  ✅ [5/16] Wikipedia — success
  ❌ [6/16] Gates Notes — failed_download


  ✅ [7/16] ai.google — success
  ✅ [8/16] OpenAI — success
  ✅ [9/16] Copyright Office (.gov) — success


...

  ✅ [12/16] The Regulatory Review — success
  ✅ [13/16] The University of Virginia — success


  ✅ [14/16] University of Southern California — success


  ❌ [15/16] Congress.gov — failed_download
  ✅ [16/16] National Institutes of Health (NIH) | (.gov) — success

  📊 Articles: 10/16 scraped successfully



🧹 Removed 4 off-topic pages:
  ❌ Wikipedia — Artificial intelligence (0.46 topic mentions per 1,000 words)
  ❌ Google Cloud — What is Artificial Intelligence (AI)? (0.0 topic mentions per 1,000 words)
  ❌ OpenAI — OpenAI | Research & Deployment (0.0 topic mentions per 1,000 words)
  ❌ ai.google — Google AI - How we're making AI helpful for everyone (0.0 topic mentions per 1,000 words)

📊 Kept 6 on-topic documents


,query,source,title,topic_mentions_per_1k
0,copyright Generative AI,Copyright Office (.gov),Copyright and Artificial Intelligence | U.S. C...,60.00
1,copyright Generative AI,The Regulatory Review,Copyright and Generative AI,33.80
2,copyright Generative AI,University of South Florida,AI Tools and Resources: Copyright and Generati...,51.00
3,copyright Generative AI,University of Southern California,"AI, Copyright, and the Law: The Ongoing Battle...",49.90
4,copyright Generative AI,The University of Virginia,Copyright and Intellectual Property - Generati...,64.79
5,copyright Generative AI,National Institutes of Health (NIH) | (.gov),The Law and Economics of Generative AI and Cop...,20.68


In [8]:
df_clean

,query,title,snippet,source,date,url,type,full_text,video_id,status,topic_mentions_per_1k
0,copyright Generative AI,Copyright and Artificial Intelligence | U.S. C...,Copyright and Artificial Intelligence analyzes...,Copyright Office (.gov),NaN,https://www.copyright.gov/ai/,article,Copyright and Artificial Intelligence\nSince l...,None,success,60.00
1,copyright Generative AI,Copyright and Generative AI,The legal framework governing copyright faces ...,The Regulatory Review,"Jun 7, 2025",https://www.theregreview.org/2025/06/07/semina...,article,Scholars discuss emerging problems at the inte...,None,success,33.80
2,copyright Generative AI,AI Tools and Resources: Copyright and Generati...,Generative AI tools can be used to infringe on...,University of South Florida,"Oct 14, 2026",https://guides.lib.usf.edu/AI/copyright,article,Copyright is a form of intellectual property p...,None,success,51.00
3,copyright Generative AI,"AI, Copyright, and the Law: The Ongoing Battle...",Bills like the Generative AI Copyright Disclos...,University of Southern California,"Feb 4, 2025",https://sites.usc.edu/iptls/2025/02/04/ai-copy...,article,By: Negar Bondari\nArtificial intelligence (AI...,None,success,49.90
4,copyright Generative AI,Copyright and Intellectual Property - Generati...,This guide features links and information abou...,The University of Virginia,NaN,https://guides.lib.virginia.edu/genai/copyright,article,Connect with our knowledgeable library staff a...,None,success,64.79
5,copyright Generative AI,The Law and Economics of Generative AI and Cop...,Generative AI challenges copyright law at both...,National Institutes of Health (NIH) | (.gov),NaN,https://pmc.ncbi.nlm.nih.gov/articles/PMC12658...,article,Abstract\nGenerative AI (GenAI) systems raise ...,None,success,20.68


In [9]:
df_clean.to_csv("ai_copyright_dataset.csv", index=False)